# Neural Network Analysis for AI Lab Gene Expression Data

This notebook uses neural networks to study the SmartSeq and DropSeq gene expression data.

The goal is not only to get high accuracy, but also to learn useful patterns from the data.

We will use:
- MLP classifier
- Autoencoder
- Denoising autoencoder
- 1D CNN classifier
- simple cross-dataset transfer test

The notebook saves:
- model results
- predictions
- latent embeddings
- important genes

## 1. Install packages if needed

Run this only if some packages are missing.

In [ ]:
# !pip install torch numpy pandas matplotlib seaborn scikit-learn umap-learn

   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 1.0/114.5 MB 5.0 MB/s eta 0:00:23
    --------------------------------------- 1.6/114.5 MB 3.7 MB/s eta 0:00:31
   - -------------------------------------- 2.9/114.5 MB 4.8 MB/s eta 0:00:24
   - -------------------------------------- 5.0/114.5 MB 5.9 MB/s eta 0:00:19
   -- ------------------------------------- 7.1/114.5 MB 6.8 MB/s eta 0:00:16
   --- ------------------------------------ 8.9/114.5 MB 7.3 MB/s eta 0:00:15
   --- ------------------------------------ 10.7/114.5 MB 7.6 MB/s eta 0:00:14
   ---- ----------------------------------- 12.3/114.5 MB 7.7 MB/s eta 0:00:14
   ---- ----------------------------------- 13.6/114.5 MB 7.4 MB/s eta 0:00:14
   ----- ---------------------------------- 14.9/114.5 MB 7.3 MB/s eta 0:00:14
   ----- ---------------------------------- 16.8/114.5 MB 7.5 MB/s eta 0:00:14
   ------ --------------------------------- 18.9/114.5 MB 7.7 MB/s

In [6]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 60.4 MB/s eta 0:00:46
   ---------------------------------------- 0.0/2.8 GB 65.1 MB/s eta 0:00:43
    --------------------------------------- 0.0/2.8 GB 67.9 MB/s eta 0:00:41
    --------------------------------------- 0.1/2.8 GB 68.0 MB/s eta 0:00:40
   - -------------------------------------- 0.1/2.8 GB 67.3 MB/s eta 0:00:40
   - -------------------------------------- 0.1/2.8 GB 67.1 MB/s eta 0:00:40
   - -------------------------------------- 0.1/2.8 GB 64.5 MB/s eta 0:00:42
   - -------------------------------------- 0.1/2.8 GB 63.9 MB/s eta 0:00:42
   - -------------------------------------- 0.1/2.8 GB 64.9 MB/s eta 0:00:41
   - -------------------------------------- 0.1/2.8 GB 65.6 MB/s eta 0:00:41
   -- ------------------------------------- 0.1/2.8 GB 66.2 MB/s eta 0:00:40
   -- -----------

## 2. Imports

In [8]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import umap

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cpu


In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


AssertionError: Torch not compiled with CUDA enabled

## 3. Set paths

Change `BASE` to your own data folder.

In [ ]:
BASE = Path(r"C:\Users\hugoo\OneDrive\Desktop\ai-lab-report\data")

SMART = BASE / "SmartSeq"
DROP = BASE / "DropSeq"

OUTPUT = BASE.parent / "nn_outputs"
OUTPUT.mkdir(exist_ok=True)

print("SmartSeq exists:", SMART.exists())
print("DropSeq exists:", DROP.exists())
print("Output folder:", OUTPUT)

## 4. Helper functions to load the data

In [ ]:
def load_expr(path):
    df = pd.read_csv(path, sep=r"\s+", engine="python", index_col=0)
    df.index = df.index.astype(str).str.replace('"', '', regex=False)
    df.columns = df.columns.astype(str).str.replace('"', '', regex=False)
    return df

def get_dropseq_condition(sample_name):
    if "Normoxia" in sample_name:
        return "Normoxia"
    if "Hypoxia" in sample_name:
        return "Hypoxia"
    return "Unknown"

def make_binary_labels(labels):
    return labels.astype(str).str.contains("Hyp", case=False, na=False).astype(int)

def prepare_xy(expr, labels):
    X = expr.T.copy()
    y = make_binary_labels(labels.loc[X.index])
    return X, y

## 5. Load normalized training data

The neural networks use the normalized 3000-gene training files.

In [ ]:
smart_mcf7 = load_expr(SMART / "MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt")
smart_hcc = load_expr(SMART / "HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt")

drop_mcf7 = load_expr(DROP / "MCF7_Filtered_Normalised_3000_Data_train.txt")
drop_hcc = load_expr(DROP / "HCC1806_Filtered_Normalised_3000_Data_train.txt")

meta_mcf7 = pd.read_csv(SMART / "MCF7_SmartS_MetaData.tsv", sep="\t")
meta_hcc = pd.read_csv(SMART / "HCC1806_SmartS_MetaData.tsv", sep="\t")

meta_mcf7["Filename"] = meta_mcf7["Filename"].astype(str).str.replace('"', '', regex=False)
meta_hcc["Filename"] = meta_hcc["Filename"].astype(str).str.replace('"', '', regex=False)

smart_mcf7_labels = meta_mcf7.set_index("Filename").loc[smart_mcf7.columns, "Condition"]
smart_hcc_labels = meta_hcc.set_index("Filename").loc[smart_hcc.columns, "Condition"]

drop_mcf7_labels = pd.Series([get_dropseq_condition(c) for c in drop_mcf7.columns], index=drop_mcf7.columns)
drop_hcc_labels = pd.Series([get_dropseq_condition(c) for c in drop_hcc.columns], index=drop_hcc.columns)

datasets = {
    "SmartSeq_MCF7": (smart_mcf7, smart_mcf7_labels),
    "SmartSeq_HCC1806": (smart_hcc, smart_hcc_labels),
    "DropSeq_MCF7": (drop_mcf7, drop_mcf7_labels),
    "DropSeq_HCC1806": (drop_hcc, drop_hcc_labels),
}

for name, (expr, labels) in datasets.items():
    print(name, expr.shape, labels.value_counts().to_dict())

## 6. Train/test split helper

This function scales the genes and creates PyTorch data loaders.

In [ ]:
def make_loaders(expr, labels, batch_size=32, test_size=0.2):
    X, y = prepare_xy(expr, labels)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=test_size,
        random_state=SEED,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    train_ds = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    )

    val_ds = TensorDataset(
        torch.tensor(X_val_scaled, dtype=torch.float32),
        torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, scaler, X_train, X_val, y_train, y_val

## 7. Training helper

This is shared by all binary classification neural networks.

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs=40, lr=1e-3):
    model = model.to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        preds = []
        trues = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                probs = torch.sigmoid(logits)

                val_losses.append(loss.item())
                preds.extend((probs.cpu().numpy().ravel() >= 0.5).astype(int))
                trues.extend(yb.cpu().numpy().ravel().astype(int))

        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds)

        history.append({
            "epoch": epoch,
            "train_loss": np.mean(train_losses),
            "val_loss": np.mean(val_losses),
            "val_accuracy": acc,
            "val_f1": f1
        })

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | train loss {np.mean(train_losses):.4f} | val acc {acc:.3f} | val f1 {f1:.3f}")

    return pd.DataFrame(history)

def evaluate_classifier(model, val_loader):
    model.eval()
    preds = []
    probs_all = []
    trues = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            probs_all.extend(probs)
            preds.extend((probs >= 0.5).astype(int))
            trues.extend(yb.numpy().ravel().astype(int))

    print(classification_report(trues, preds))
    print("Confusion matrix:")
    print(confusion_matrix(trues, preds))

    return pd.DataFrame({
        "true": trues,
        "pred": preds,
        "prob_hypoxia": probs_all
    })

## 8. Model 1: MLP classifier

This is the main neural network baseline. It uses fully connected layers to classify samples as normoxia or hypoxia.

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)

## 9. Run MLP on one dataset

Change `DATASET_NAME` to try the others.

In [ ]:
DATASET_NAME = "SmartSeq_MCF7"

expr, labels = datasets[DATASET_NAME]
train_loader, val_loader, scaler, X_train, X_val, y_train, y_val = make_loaders(expr, labels)

mlp = MLPClassifier(input_dim=X_train.shape[1])
mlp_history = train_classifier(mlp, train_loader, val_loader, epochs=40, lr=1e-3)

mlp_preds = evaluate_classifier(mlp, val_loader)

mlp_history.to_csv(OUTPUT / f"{DATASET_NAME}_mlp_history.csv", index=False)
mlp_preds.to_csv(OUTPUT / f"{DATASET_NAME}_mlp_predictions.csv", index=False)

## 10. Plot MLP training curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(mlp_history["epoch"], mlp_history["train_loss"], label="train loss")
plt.plot(mlp_history["epoch"], mlp_history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{DATASET_NAME} - MLP loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(mlp_history["epoch"], mlp_history["val_accuracy"], label="val accuracy")
plt.plot(mlp_history["epoch"], mlp_history["val_f1"], label="val F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title(f"{DATASET_NAME} - MLP validation performance")
plt.legend()
plt.show()

## 11. Model 2: Autoencoder

An autoencoder learns a compressed representation of the gene expression data.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

    def encode(self, x):
        return self.encoder(x)

## 12. Train autoencoder

In [ ]:
def train_autoencoder(expr, labels, latent_dim=16, epochs=50, lr=1e-3):
    X, y = prepare_xy(expr, labels)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ds = TensorDataset(torch.tensor(X_scaled, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=32, shuffle=True)

    model = Autoencoder(input_dim=X_scaled.shape[1], latent_dim=latent_dim).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for (xb,) in loader:
            xb = xb.to(DEVICE)
            optimizer.zero_grad()
            recon = model(xb)
            loss = loss_fn(recon, xb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        history.append({"epoch": epoch, "reconstruction_loss": np.mean(losses)})

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | reconstruction loss {np.mean(losses):.4f}")

    return model, scaler, pd.DataFrame(history), X, y

ae_model, ae_scaler, ae_history, ae_X, ae_y = train_autoencoder(expr, labels, latent_dim=16, epochs=50)
ae_history.to_csv(OUTPUT / f"{DATASET_NAME}_autoencoder_history.csv", index=False)

## 13. Plot autoencoder latent space with UMAP

This shows the neural network representation in 2D.

In [ ]:
def plot_autoencoder_latent(model, scaler, X, y, title):
    X_scaled = scaler.transform(X)

    model.eval()
    with torch.no_grad():
        z = model.encode(torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)).cpu().numpy()

    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED)
    emb = reducer.fit_transform(z)

    plot_df = pd.DataFrame({
        "UMAP1": emb[:, 0],
        "UMAP2": emb[:, 1],
        "condition": np.where(y.values == 1, "Hypoxia", "Normoxia")
    }, index=X.index)

    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=plot_df, x="UMAP1", y="UMAP2", hue="condition", s=40)
    plt.title(title)
    plt.show()

    return pd.DataFrame(z, index=X.index).add_prefix("latent_")

latent_df = plot_autoencoder_latent(
    ae_model,
    ae_scaler,
    ae_X,
    ae_y,
    f"{DATASET_NAME} - Autoencoder latent space"
)

latent_df.to_csv(OUTPUT / f"{DATASET_NAME}_autoencoder_latent.csv")

## 14. Model 3: Denoising autoencoder

This adds noise to the input and learns to reconstruct the clean data.

In [ ]:
def train_denoising_autoencoder(expr, labels, latent_dim=16, noise_std=0.2, epochs=50, lr=1e-3):
    X, y = prepare_xy(expr, labels)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ds = TensorDataset(torch.tensor(X_scaled, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=32, shuffle=True)

    model = Autoencoder(input_dim=X_scaled.shape[1], latent_dim=latent_dim).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for (xb,) in loader:
            xb = xb.to(DEVICE)
            noisy_xb = xb + noise_std * torch.randn_like(xb)

            optimizer.zero_grad()
            recon = model(noisy_xb)
            loss = loss_fn(recon, xb)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        history.append({"epoch": epoch, "denoising_loss": np.mean(losses)})

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | denoising loss {np.mean(losses):.4f}")

    return model, scaler, pd.DataFrame(history), X, y

dae_model, dae_scaler, dae_history, dae_X, dae_y = train_denoising_autoencoder(expr, labels)
dae_history.to_csv(OUTPUT / f"{DATASET_NAME}_denoising_autoencoder_history.csv", index=False)

## 15. Plot denoising autoencoder latent space

In [ ]:
dae_latent_df = plot_autoencoder_latent(
    dae_model,
    dae_scaler,
    dae_X,
    dae_y,
    f"{DATASET_NAME} - Denoising autoencoder latent space"
)

dae_latent_df.to_csv(OUTPUT / f"{DATASET_NAME}_denoising_autoencoder_latent.csv")

## 16. Model 4: 1D CNN classifier

This treats the gene vector like a 1D signal. This is not biologically perfect because genes do not have a natural image-like order, but it is useful as a model comparison.

In [ ]:
class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(64)
        )

        self.fc = nn.Sequential(
            nn.Linear(32 * 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)

## 17. Train 1D CNN

In [ ]:
cnn = CNN1DClassifier(input_dim=X_train.shape[1])
cnn_history = train_classifier(cnn, train_loader, val_loader, epochs=40, lr=1e-3)
cnn_preds = evaluate_classifier(cnn, val_loader)

cnn_history.to_csv(OUTPUT / f"{DATASET_NAME}_cnn_history.csv", index=False)
cnn_preds.to_csv(OUTPUT / f"{DATASET_NAME}_cnn_predictions.csv", index=False)

## 18. Model comparison table

In [ ]:
def final_scores(pred_df, model_name):
    return {
        "model": model_name,
        "accuracy": accuracy_score(pred_df["true"], pred_df["pred"]),
        "f1": f1_score(pred_df["true"], pred_df["pred"])
    }

comparison = pd.DataFrame([
    final_scores(mlp_preds, "MLP"),
    final_scores(cnn_preds, "1D CNN")
])

comparison.to_csv(OUTPUT / f"{DATASET_NAME}_nn_model_comparison.csv", index=False)
comparison

## 19. Gene importance with gradients

This checks which genes the MLP is most sensitive to.

In [ ]:
def gradient_gene_importance(model, scaler, X, top_n=25):
    X_scaled = scaler.transform(X)
    x_tensor = torch.tensor(X_scaled, dtype=torch.float32, requires_grad=True).to(DEVICE)

    model.eval()
    logits = model(x_tensor)
    score = logits.mean()
    score.backward()

    grads = x_tensor.grad.detach().cpu().numpy()
    importance = np.abs(grads).mean(axis=0)

    imp = pd.Series(importance, index=X.columns).sort_values(ascending=False)

    plt.figure(figsize=(8, 6))
    imp.head(top_n).sort_values().plot(kind="barh")
    plt.title("Top genes by gradient importance")
    plt.xlabel("Average absolute gradient")
    plt.show()

    return imp

gene_importance = gradient_gene_importance(mlp, scaler, X_val, top_n=25)
gene_importance.to_csv(OUTPUT / f"{DATASET_NAME}_mlp_gradient_gene_importance.csv")
gene_importance.head(25)

## 20. Cross-dataset transfer test

This tests whether a model trained on one technology can work on another technology.

In [ ]:
def transfer_test(train_expr, train_labels, test_expr, test_labels, name):
    common_genes = train_expr.index.intersection(test_expr.index)

    train_expr = train_expr.loc[common_genes]
    test_expr = test_expr.loc[common_genes]

    X_train, y_train = prepare_xy(train_expr, train_labels)
    X_test, y_test = prepare_xy(test_expr, test_labels)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    train_ds = TensorDataset(
        torch.tensor(X_train_scaled, dtype=torch.float32),
        torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    )
    test_ds = TensorDataset(
        torch.tensor(X_test_scaled, dtype=torch.float32),
        torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
    )

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    model = MLPClassifier(input_dim=len(common_genes))
    _ = train_classifier(model, train_loader, test_loader, epochs=30, lr=1e-3)
    preds = evaluate_classifier(model, test_loader)

    result = {
        "experiment": name,
        "accuracy": accuracy_score(preds["true"], preds["pred"]),
        "f1": f1_score(preds["true"], preds["pred"]),
        "n_common_genes": len(common_genes)
    }

    return result, preds

transfer_results = []

res, preds = transfer_test(
    smart_mcf7, smart_mcf7_labels,
    drop_mcf7, drop_mcf7_labels,
    "Train SmartSeq MCF7 -> Test DropSeq MCF7"
)
transfer_results.append(res)
preds.to_csv(OUTPUT / "transfer_smart_mcf7_to_drop_mcf7_predictions.csv", index=False)

res, preds = transfer_test(
    drop_mcf7, drop_mcf7_labels,
    smart_mcf7, smart_mcf7_labels,
    "Train DropSeq MCF7 -> Test SmartSeq MCF7"
)
transfer_results.append(res)
preds.to_csv(OUTPUT / "transfer_drop_mcf7_to_smart_mcf7_predictions.csv", index=False)

transfer_df = pd.DataFrame(transfer_results)
transfer_df.to_csv(OUTPUT / "transfer_results.csv", index=False)
transfer_df

## 21. Final neural network summary

Use this section to write your interpretation.

Things to discuss:
- Did the MLP classify hypoxia well?
- Did the CNN improve results or not?
- Did the autoencoder latent space separate conditions?
- Did transfer learning work across technologies?
- If transfer performance dropped, that suggests technology/batch effects are strong.